# GPS OD-Flow Training

Train GPS models with different configurations. Weights and metrics are saved to `results/`
and consumed by `benchmark.ipynb` for comparison.

**Architecture options:**
- `encoder_type`: `'gps'` (GPS-GNN) | `'mlp'` (TransFlower)
- `pe_type`: `'rwpe'` | `'spe'` | `'rrwp'` | `'lape'` (WeDAN dual Laplacian)
- `loss_type`: `'huber'` | `'ce'` | `'multitask'` | `'zinb'` | `'focal'`
- `use_rle`: adds Relative Location Encoder


In [ ]:
import sys, os, gc, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

sys.path.insert(0, str(Path('.').resolve()))
warnings.filterwarnings('ignore')

from models.GPS.config import (
    TrainingConfig, device, MC_EPOCHS,
    SINGLE_CITY_ID, MULTI_CITY_IDS, RESULTS_DIR, METRICS_CSV, WEIGHTS_DIR,
    ensure_dirs,
)
from models.GPS.data_load import prepare_single_city_data, prepare_multi_city_data
from models.GPS.main import train_single_city, train_multi_city
from models.GPS.metrics import evaluate_full_matrix, compute_metrics

ensure_dirs()
print(f"Device: {device}")
print(f"Results dir: {RESULTS_DIR}")


## GPS Configurations

Edit this cell to select which experiments to run.

In [ ]:
from dataclasses import replace

# ── Single-city run configurations ──────────────────────────────────────────
# Format: run_id -> (display_name, TrainingConfig)
SC_CONFIGS = {
    # ─── GPS + TransFlower decoder ───────────────────────────────────────────
    'SC_TF_CE':          ('SC TF CE: GPS+TF+CE (norm, no samp)',      TrainingConfig(decoder_type='transflower', loss_type='ce',        prediction_mode='normalized', use_dest_sampling=False)),
    'SC_TF_H':           ('SC TF H: GPS+TF+Huber (raw)',              TrainingConfig(decoder_type='transflower', loss_type='huber',     prediction_mode='raw')),
    'SC_TF_CE_spe':      ('SC TF CE spe: +pe=spe',                   TrainingConfig(decoder_type='transflower', loss_type='ce',        prediction_mode='normalized', use_dest_sampling=False, pe_type='spe')),
    'SC_TF_CE_rrwp':     ('SC TF CE rrwp: +pe=rrwp',                 TrainingConfig(decoder_type='transflower', loss_type='ce',        prediction_mode='normalized', use_dest_sampling=False, pe_type='rrwp')),
    'SC_TF_CE_lape':     ('SC TF CE lape: +pe=lape',                 TrainingConfig(decoder_type='transflower', loss_type='ce',        prediction_mode='normalized', use_dest_sampling=False, pe_type='lape')),
    'SC_TF_CE_gn':       ('SC TF CE gn: +norm=graph_norm',           TrainingConfig(decoder_type='transflower', loss_type='ce',        prediction_mode='normalized', use_dest_sampling=False, gps_norm_type='graph_norm')),
    'SC_TF_multitask':   ('SC TF multitask: GPS+TF+Multitask (raw)', TrainingConfig(decoder_type='transflower', loss_type='multitask', prediction_mode='raw')),
    'SC_TF_focal':       ('SC TF focal: GPS+TF+Focal (norm)',         TrainingConfig(decoder_type='transflower', loss_type='focal',     prediction_mode='normalized', use_dest_sampling=False)),
    'SC_TF_CE_samp':     ('SC TF CE samp: +sampling zeros30%',       TrainingConfig(decoder_type='transflower', loss_type='ce',        prediction_mode='normalized', use_dest_sampling=True, n_dest_sample=128, include_zero_pairs=True, zero_pair_ratio=0.3)),
    'SC_TF_CE_nz':       ('SC TF CE nz: +sampling no-zeros',         TrainingConfig(decoder_type='transflower', loss_type='ce',        prediction_mode='normalized', use_dest_sampling=True, n_dest_sample=128, include_zero_pairs=False)),
    'SC_TF_H_log':       ('SC TF H log: Huber+raw+log',              TrainingConfig(decoder_type='transflower', loss_type='huber',     prediction_mode='raw', use_log_transform=True)),
    'SC_TF_CE_log':      ('SC TF CE log: CE+norm+log',               TrainingConfig(decoder_type='transflower', loss_type='ce',        prediction_mode='normalized', use_dest_sampling=False, use_log_transform=True)),
    'SC_TF_CE_rle':      ('SC TF CE rle: +use_rle',                  TrainingConfig(decoder_type='transflower', loss_type='ce',        prediction_mode='normalized', use_dest_sampling=False, use_rle=True)),
    'SC_TF_CE_lape_rle': ('SC TF CE lape+rle: +lape+rle',           TrainingConfig(decoder_type='transflower', loss_type='ce',        prediction_mode='normalized', use_dest_sampling=False, pe_type='lape', use_rle=True)),
    'SC_TF_focal_rle':   ('SC TF focal+rle: Focal+rle',             TrainingConfig(decoder_type='transflower', loss_type='focal',     prediction_mode='normalized', use_dest_sampling=False, use_rle=True)),
    # ─── GPS + Bilinear decoder ──────────────────────────────────────────────
    'SC_BL_CE':          ('SC BL CE: GPS+BL+CE (norm, no samp)',      TrainingConfig(decoder_type='bilinear',    loss_type='ce',        prediction_mode='normalized', use_dest_sampling=False)),
    'SC_BL_H':           ('SC BL H: GPS+BL+Huber (raw)',              TrainingConfig(decoder_type='bilinear',    loss_type='huber',     prediction_mode='raw')),
    'SC_BL_CE_spe':      ('SC BL CE spe: +pe=spe',                   TrainingConfig(decoder_type='bilinear',    loss_type='ce',        prediction_mode='normalized', use_dest_sampling=False, pe_type='spe')),
    'SC_BL_CE_rrwp':     ('SC BL CE rrwp: +pe=rrwp',                 TrainingConfig(decoder_type='bilinear',    loss_type='ce',        prediction_mode='normalized', use_dest_sampling=False, pe_type='rrwp')),
    'SC_BL_CE_lape':     ('SC BL CE lape: +pe=lape',                 TrainingConfig(decoder_type='bilinear',    loss_type='ce',        prediction_mode='normalized', use_dest_sampling=False, pe_type='lape')),
    'SC_BL_CE_gn':       ('SC BL CE gn: +norm=graph_norm',           TrainingConfig(decoder_type='bilinear',    loss_type='ce',        prediction_mode='normalized', use_dest_sampling=False, gps_norm_type='graph_norm')),
    'SC_BL_multitask':   ('SC BL multitask: GPS+BL+Multitask (raw)', TrainingConfig(decoder_type='bilinear',    loss_type='multitask', prediction_mode='raw')),
    'SC_BL_focal':       ('SC BL focal: GPS+BL+Focal (norm)',         TrainingConfig(decoder_type='bilinear',    loss_type='focal',     prediction_mode='normalized', use_dest_sampling=False)),
    'SC_BL_CE_samp':     ('SC BL CE samp: +sampling zeros30%',       TrainingConfig(decoder_type='bilinear',    loss_type='ce',        prediction_mode='normalized', use_dest_sampling=True, n_dest_sample=128, include_zero_pairs=True, zero_pair_ratio=0.3)),
    'SC_BL_CE_nz':       ('SC BL CE nz: +sampling no-zeros',         TrainingConfig(decoder_type='bilinear',    loss_type='ce',        prediction_mode='normalized', use_dest_sampling=True, n_dest_sample=128, include_zero_pairs=False)),
    'SC_BL_H_log':       ('SC BL H log: Huber+raw+log',              TrainingConfig(decoder_type='bilinear',    loss_type='huber',     prediction_mode='raw', use_log_transform=True)),
    'SC_BL_CE_log':      ('SC BL CE log: CE+norm+log',               TrainingConfig(decoder_type='bilinear',    loss_type='ce',        prediction_mode='normalized', use_dest_sampling=False, use_log_transform=True)),
}

# ── Multi-city run configurations ─────────────────────────────────────────────
# Mirror of SC_CONFIGS with MC_ prefix and mc_epochs set
MC_CONFIGS = {
    rid.replace('SC_', 'MC_'): (name.replace('SC ', 'MC '), replace(cfg, mc_epochs=MC_EPOCHS))
    for rid, (name, cfg) in SC_CONFIGS.items()
}

print(f"Single-city configs: {len(SC_CONFIGS)}")
print(f"Multi-city configs:  {len(MC_CONFIGS)}")


## Single-City Training

In [ ]:
# ── Data loading with pe_type caching ────────────────────────────────────────
_sc_cache = {}

def get_sc_data(pe_type='rwpe'):
    if pe_type not in _sc_cache:
        print(f"  Loading single-city data (pe_type={pe_type})...")
        _sc_cache[pe_type] = prepare_single_city_data(pe_type=pe_type)
        N = _sc_cache[pe_type]['num_nodes']
        tm = _sc_cache[pe_type]['train_mask'].sum()
        print(f"  N={N}, train_pairs={tm}")
    return _sc_cache[pe_type]


In [ ]:
sc_results = {}

for run_id, (run_name, cfg) in SC_CONFIGS.items():
    city_data = get_sc_data(cfg.pe_type)
    try:
        result = train_single_city(run_id, run_name, cfg, city_data=city_data)
        sc_results[run_id] = result
    except Exception as e:
        print(f"  ERROR {run_id}: {e}")
    finally:
        if device.type == 'cuda':
            torch.cuda.empty_cache()
            gc.collect()

print(f"\nCompleted: {len(sc_results)} single-city runs")


## Multi-City Training

In [ ]:
# ── Multi-city data loading with pe_type caching ─────────────────────────────
_mc_cache = {}

def get_mc_data(pe_type='rwpe'):
    if pe_type not in _mc_cache:
        print(f"  Loading multi-city data (pe_type={pe_type})...")
        city_data_dict, train_ids, val_ids, test_ids = prepare_multi_city_data(pe_type=pe_type)
        _mc_cache[pe_type] = (city_data_dict, train_ids, val_ids, test_ids)
        print(f"  Train: {train_ids}  Val: {val_ids}  Test: {test_ids}")
    return _mc_cache[pe_type]


In [ ]:
mc_results = {}

for run_id, (run_name, cfg) in MC_CONFIGS.items():
    city_data_dict, train_ids, val_ids, test_ids = get_mc_data(cfg.pe_type)
    try:
        result = train_multi_city(
            run_id, run_name, cfg,
            city_data_dict=city_data_dict,
            train_city_ids=train_ids,
            val_city_ids=val_ids,
        )
        mc_results[run_id] = result
    except Exception as e:
        print(f"  ERROR {run_id}: {e}")
    finally:
        if device.type == 'cuda':
            torch.cuda.empty_cache()
            gc.collect()

print(f"\nCompleted: {len(mc_results)} multi-city runs")


## Results Summary

In [ ]:
def print_results_table(results, label):
    if not results:
        print(f"  No {label} results.")
        return
    print(f"\n{'='*70}")
    print(f"  {label} Results")
    print(f"{'='*70}")
    print(f"  {'Run ID':<18} {'CPC_full':>9} {'CPC_nz':>9} {'CPC_test':>9} {'MAE':>9}  Status")
    print(f"  {'-'*66}")
    for rid, r in results.items():
        mf = r.get('metrics_full', {})
        mnz = r.get('metrics_nonzero', {})
        mt = r.get('metrics_test_pairs', {})
        status = r.get('status', '?')
        print(f"  {rid:<18} {mf.get('CPC', float('nan')):>9.4f} "
              f"{mnz.get('CPC', float('nan')):>9.4f} "
              f"{mt.get('CPC', float('nan')):>9.4f} "
              f"{mf.get('MAE', float('nan')):>9.4f}  {status}")

print_results_table(sc_results, "Single-City")
print_results_table(mc_results, "Multi-City")

# Show saved metrics CSV
if METRICS_CSV.exists():
    df = pd.read_csv(METRICS_CSV)
    print(f"\n  Total rows in metrics.csv: {len(df)}")
    print(f"  Saved weights: {len(list(WEIGHTS_DIR.glob('*.pt')))}")


In [ ]:
# Training curves
def plot_histories(results, title):
    if not results:
        return
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(title)
    for rid, r in results.items():
        h = r.get('history', {})
        if not h:
            continue
        axes[0].plot(h.get('train_loss', []), label=rid, alpha=0.8)
        axes[1].plot(h.get('val_loss', []), label=rid, alpha=0.8)
        axes[2].plot(h.get('val_cpc_full', []), label=rid, alpha=0.8)
    axes[0].set_title('Train Loss'); axes[0].set_xlabel('Epoch')
    axes[1].set_title('Val Loss');   axes[1].set_xlabel('Epoch')
    axes[2].set_title('CPC (full)'); axes[2].set_xlabel('Epoch')
    for ax in axes:
        ax.legend(fontsize=7, ncol=2)
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_histories(sc_results, "Single-City Training Histories")
plot_histories(mc_results, "Multi-City Training Histories")
